#### Strategy Outline

Bias for commercial net long > 80 and <20 for net short
When bias is long and RSI is <30 go long, go short when bias is short and rsi is  over 70 go short

Exit: 3 days.  Option to also exit on RSI @50 or 20 day limit

Risk management: 2 ATR stop, 3 ATR target, 1% risk per trade

In [14]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

with open('cot_data.json', 'r') as f:
    data__raw = json.load(f)

df = pd.DataFrame(data__raw)
df["Date"] = pd.to_datetime(df["Date"], unit='ms')  # Convert from milliseconds


In [23]:
# Count nulls by Market for Open column
null_counts_by_market = df.groupby('Market')['Close'].apply(lambda x: x.isnull().sum()).sort_values(ascending=False)
print("Markets with most Open price nulls:")
print(null_counts_by_market.head(10))

Markets with most Open price nulls:
Market
EURO FX - CHICAGO MERCANTILE EXCHANGE                         103
CORN - CHICAGO BOARD OF TRADE                                 102
SOYBEAN OIL - CHICAGO BOARD OF TRADE                          102
SOYBEAN MEAL - CHICAGO BOARD OF TRADE                         102
ULTRA UST BOND - CHICAGO BOARD OF TRADE                       101
NORTH EURO HOT-ROLL COIL STEEL - COMMODITY EXCHANGE INC.       83
WHEAT-HRSpring - MIAX FUTURES EXCHANGE                         56
NIKKEI STOCK AVERAGE - CHICAGO MERCANTILE EXCHANGE             54
OATS - CHICAGO BOARD OF TRADE                                  40
RUSSELL 2000 ANNUAL DIVIDEND - CHICAGO MERCANTILE EXCHANGE     39
Name: Close, dtype: int64


#### Prepare data for analysis

In [4]:
def prepare_strategy_data(df, market_name):

    market_data = df[df['Market'] == market_name].copy()

    cot_weekly = market_data[market_data['data_type'] == 'weekly_cot'].copy()
    price_daily = market_data[market_data['data_type'] == 'daily_price'].copy()

    if price_daily.empty:
        print(f"No daily price data for {market_name}, using weekly data")
        return cot_weekly

    # ENHANCED: Include OHLC data for ATR calculation and entry at open prices
    price_cols = ['Date', 'Close', 'Open', 'High', 'Low', 'RSI']
    available_cols = [col for col in price_cols if col in price_daily.columns]
    strategy_data = price_daily[available_cols].copy()
    
    print(f"Available price columns for {market_name}: {available_cols}")

    # SIMPLE FIX: Just merge weekly COT data directly, then forward fill
    cot_cols = ['Net Commercial Position', 'OI', 'Commercial_Index']
    cot_for_merge = cot_weekly[['Date'] + cot_cols].copy()
    
    strategy_data = pd.merge(strategy_data, cot_for_merge, on='Date', how='left')
    
    # Forward fill COT values to fill gaps between weekly reports
    strategy_data[cot_cols] = strategy_data[cot_cols].fillna(method='ffill')
    
    # Remove rows with missing critical data
    strategy_data = strategy_data.dropna(subset=['Close', 'Commercial_Index'])
    
    return strategy_data.sort_values('Date').reset_index(drop=True) 



In [5]:
#Confirm the function is working using  a random market 
prepare_strategy_data(df, "AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE")

Available price columns for AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE: ['Date', 'Close', 'RSI']


,Date,Close,RSI,Net Commercial Position,OI,Commercial_Index
0,2024-01-02,0.67645,NaN,30295.0,157003.0,48.004386
1,2024-01-03,0.67330,NaN,30295.0,157003.0,48.004386
2,2024-01-04,0.67055,NaN,30295.0,157003.0,48.004386
3,2024-01-05,0.67155,NaN,30295.0,157003.0,48.004386
4,2024-01-08,0.67245,NaN,30295.0,157003.0,48.004386
...,...,...,...,...,...,...
496,2025-12-19,0.66200,42.804357,1571.0,293796.0,30.885675
497,2025-12-22,0.66555,60.128562,1571.0,293796.0,30.885675
498,2025-12-23,0.66995,66.304288,1571.0,293796.0,30.885675
499,2025-12-24,0.67085,62.309908,1571.0,293796.0,30.885675


### Technical Indicators

In [6]:
# ATR (Average True Range) calculation for stop-loss and position sizing
def calculate_atr(data, period=14):
    """
    Calculate Average True Range (ATR) for volatility measurement
    """
    if 'High' in data.columns and 'Low' in data.columns and 'Close' in data.columns:
        # Use actual OHLC data for precise ATR
        high_low = data['High'] - data['Low']
        high_close_prev = abs(data['High'] - data['Close'].shift(1))
        low_close_prev = abs(data['Low'] - data['Close'].shift(1))
        
        true_range = pd.concat([high_low, high_close_prev, low_close_prev], axis=1).max(axis=1)
        atr = true_range.rolling(window=period).mean()
        
        print(f"✓ ATR calculated using actual OHLC data")
        return atr
    else:
        # Fallback: approximate ATR using close price volatility
        close_volatility = data['Close'].pct_change().abs()
        atr_approx = close_volatility.rolling(window=period).mean() * data['Close']
        
        print(f"⚠ ATR approximated using Close price volatility (OHLC not available)")
        return atr_approx


In [7]:
# Signal Generation System
def generate_signals(data):
    """
    Generate long and short signals based on Commercial Index and RSI
    
    Strategy Rules:
    - Long Signal: Commercial_Index > 70 AND RSI < 30
    - Short Signal: Commercial_Index < 21 AND RSI > 70
    """
    signals = pd.DataFrame(index=data.index)
    signals['Date'] = data['Date']
    signals['Close'] = data['Close']
    signals['Commercial_Index'] = data['Commercial_Index']
    signals['RSI'] = data['RSI']
    
    # Initialize signal columns
    signals['Signal'] = 0  # 0 = No signal, 1 = Long, -1 = Short
    signals['Position'] = 0  # Track current position
    
    # Generate signals based on strategy rules
    long_condition = (data['Commercial_Index'] > 70) & (data['RSI'] < 30)
    short_condition = (data['Commercial_Index'] < 21) & (data['RSI'] > 70)
    
    signals.loc[long_condition, 'Signal'] = 1   # Long signal
    signals.loc[short_condition, 'Signal'] = -1  # Short signal
    
    # Add Open price for next-day entry if available
    if 'Open' in data.columns:
        signals['Next_Open'] = data['Open'].shift(-1)  # Next day's open price
        signals['Entry_Price'] = signals['Next_Open']  # Entry at next day's open
    else:
        signals['Entry_Price'] = data['Close']  # Fallback to close price
    
    # Add ATR for stop-loss calculation
    signals['ATR'] = calculate_atr(data)
    
    print(f"Generated {len(signals[signals['Signal'] != 0])} total signals:")
    print(f"  - Long signals: {len(signals[signals['Signal'] == 1])}")
    print(f"  - Short signals: {len(signals[signals['Signal'] == -1])}")
    
    return signals


In [8]:
# UPDATED COT + RSI Backtesting Engine with Proper Position Sizing
class COTRSIBacktester:
    """
    Backtesting engine for COT + RSI strategy with 3-day fixed exit
    Updated with proper position sizing and error reporting
    """
    
    def __init__(self, initial_capital=100000, risk_per_trade=0.01, use_stops=True):
        self.initial_capital = initial_capital
        self.risk_per_trade = risk_per_trade  # 1% risk per trade
        self.use_stops = use_stops
        self.trades = []
        self.equity_curve = []
        
    def calculate_position_size(self, entry_price, atr, direction, row_index=None):
        """
        Calculate position size based on 1% risk per trade
        Returns integer units to avoid fractional trades
        
        Formula:
        - Stop price = entry_price - (2 * ATR * direction)
        - Position value = risked_amount / (1 - (stop_price / entry_price))
        - Position units = floor(position_value / entry_price)
        """
        
        # Check for missing ATR data
        if pd.isna(atr) or atr == 0:
            # Allow missing ATR for first 14 days (ATR calculation period)
            if row_index is not None and row_index >= 14:
                raise ValueError(f"❌ MISSING ATR DATA at row {row_index}! "
                               f"Entry price: ${entry_price:.4f}, ATR: {atr}. "
                               f"Check OHLC data quality - High/Low prices may be missing.")
            elif row_index is not None and row_index < 14:
                print(f"⚠ ATR not available for row {row_index} (within first 14 days) - SKIPPING TRADE")
                return None  # Skip this trade
            else:
                raise ValueError(f"❌ MISSING ATR DATA! Entry price: ${entry_price:.4f}, ATR: {atr}. "
                               f"Check if OHLC data (High/Low) is available in your JSON file.")
        
        # Calculate stop price based on direction
        if direction == 1:  # Long position
            stop_price = entry_price - (2 * atr)
        else:  # Short position  
            stop_price = entry_price + (2 * atr)
        
        # Ensure stop price is positive and makes sense
        if stop_price <= 0:
            raise ValueError(f"❌ INVALID STOP PRICE: ${stop_price:.4f} "
                            f"(Entry: ${entry_price:.4f}, ATR: {atr:.4f}, Direction: {direction})")
        
        # Calculate risked amount (1% of capital)
        risked_amount = self.initial_capital * self.risk_per_trade
        
        # Calculate position value using your formula
        risk_ratio = 1 - (stop_price / entry_price)
        
        if risk_ratio <= 0:
            raise ValueError(f"❌ INVALID RISK RATIO: {risk_ratio:.4f} "
                            f"(Entry: ${entry_price:.4f}, Stop: ${stop_price:.4f})")
        
        position_value = risked_amount / risk_ratio
        
        # Calculate integer position units
        position_units = int(position_value / entry_price)
        
        # Ensure minimum 1 unit
        position_units = max(1, position_units)
        
        print(f"✓ Entry: ${entry_price:.4f}, Stop: ${stop_price:.4f}, Risk: ${risked_amount:.2f}, Units: {position_units}")
        
        return position_units
    
    def backtest_strategy(self, data, signals):
        """
        Execute backtest with 3-day fixed exit and optional ATR stops
        """
        self.trades = []
        current_capital = self.initial_capital
        
        i = 0
        while i < len(signals):
            if signals.iloc[i]['Signal'] != 0:
                # Found a signal - enter trade
                entry_signal = signals.iloc[i]
                entry_price = entry_signal['Entry_Price']
                
                if pd.isna(entry_price):
                    i += 1
                    continue
                    
                direction = entry_signal['Signal']  # 1 for long, -1 for short
                atr = entry_signal['ATR']
                
                # Calculate position size with row index for error reporting
                try:
                    position_size = self.calculate_position_size(entry_price, atr, direction, row_index=i)
                    
                    # Skip trade if position_size is None (early days without ATR)
                    if position_size is None:
                        i += 1
                        continue
                        
                except ValueError as e:
                    print(f"❌ Position sizing error at row {i}: {e}")
                    i += 1
                    continue
                
                # Set stop-loss and take-profit levels (optional)
                if self.use_stops and not pd.isna(atr):
                    stop_loss = entry_price - (2 * atr * direction)  # 2 ATR stop
                    take_profit = entry_price + (3 * atr * direction)  # 3 ATR target
                else:
                    stop_loss = None
                    take_profit = None
                
                # Look for exit (3 days later or stop/TP hit)
                exit_idx = min(i + 3, len(data) - 1)  # Exit after 3 days
                exit_price = None
                exit_reason = "3-day limit"
                
                # Check for stop/TP hits during the 3-day period
                if self.use_stops and stop_loss and take_profit:
                    for j in range(i + 1, exit_idx + 1):
                        if j >= len(data):
                            break
                            
                        day_high = data.iloc[j].get('High', data.iloc[j]['Close'])
                        day_low = data.iloc[j].get('Low', data.iloc[j]['Close'])
                        
                        # Check stop-loss hit
                        if direction == 1 and day_low <= stop_loss:  # Long position
                            exit_price = stop_loss
                            exit_reason = "Stop Loss"
                            break
                        elif direction == -1 and day_high >= stop_loss:  # Short position
                            exit_price = stop_loss
                            exit_reason = "Stop Loss"
                            break
                            
                        # Check take-profit hit
                        if direction == 1 and day_high >= take_profit:  # Long position
                            exit_price = take_profit
                            exit_reason = "Take Profit"
                            break
                        elif direction == -1 and day_low <= take_profit:  # Short position
                            exit_price = take_profit
                            exit_reason = "Take Profit"
                            break
                
                # If no stop/TP hit, exit at 3-day limit
                if exit_price is None:
                    if 'Open' in data.columns and exit_idx < len(data):
                        exit_price = data.iloc[exit_idx]['Open']  # Exit at open of 4th day
                    else:
                        exit_price = data.iloc[exit_idx]['Close']
                
                # Calculate trade P&L
                if direction == 1:  # Long trade
                    pnl = (exit_price - entry_price) * position_size
                else:  # Short trade
                    pnl = (entry_price - exit_price) * position_size
                
                # Record trade
                trade = {
                    'entry_date': entry_signal['Date'],
                    'exit_date': data.iloc[exit_idx]['Date'],
                    'direction': 'Long' if direction == 1 else 'Short',
                    'entry_price': entry_price,
                    'exit_price': exit_price,
                    'position_size': position_size,
                    'pnl': pnl,
                    'exit_reason': exit_reason,
                    'commercial_index': entry_signal['Commercial_Index'],
                    'rsi': entry_signal['RSI']
                }
                
                self.trades.append(trade)
                current_capital += pnl
                
                # Record equity point
                self.equity_curve.append({
                    'date': data.iloc[exit_idx]['Date'],
                    'equity': current_capital
                })
                
                # Skip to after exit to avoid overlapping trades
                i = exit_idx + 1
            else:
                i += 1
        
        print(f"Backtest completed: {len(self.trades)} trades executed")
        return self.trades

In [ ]:
# Performance Analysis Functions
def calculate_performance_metrics(trades, initial_capital=100000):
    """
    Calculate comprehensive performance metrics
    """
    if not trades:
        return {"error": "No trades to analyze"}
    
    trades_df = pd.DataFrame(trades)
    
    # Basic metrics
    total_trades = len(trades_df)
    winning_trades = len(trades_df[trades_df['pnl'] > 0])
    losing_trades = len(trades_df[trades_df['pnl'] < 0])
    win_rate = winning_trades / total_trades if total_trades > 0 else 0
    
    # P&L metrics
    total_pnl = trades_df['pnl'].sum()
    avg_win = trades_df[trades_df['pnl'] > 0]['pnl'].mean() if winning_trades > 0 else 0
    avg_loss = trades_df[trades_df['pnl'] < 0]['pnl'].mean() if losing_trades > 0 else 0
    
    # Profit factor
    gross_profit = trades_df[trades_df['pnl'] > 0]['pnl'].sum()
    gross_loss = abs(trades_df[trades_df['pnl'] < 0]['pnl'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
    
    # Returns and Sharpe
    final_capital = initial_capital + total_pnl
    total_return = (final_capital - initial_capital) / initial_capital
    
    # Estimate annualized return (assuming trades span multiple years)
    if len(trades_df) > 0:
        start_date = pd.to_datetime(trades_df['entry_date'].min())
        end_date = pd.to_datetime(trades_df['exit_date'].max())
        years = (end_date - start_date).days / 365.25
        cagr = (final_capital / initial_capital) ** (1/years) - 1 if years > 0 else 0
    else:
        cagr = 0
    
    # Max Drawdown (simplified)
    running_pnl = trades_df['pnl'].cumsum()
    peak = running_pnl.cummax()
    drawdown = (running_pnl - peak) / initial_capital
    max_drawdown = drawdown.min()
    
    # Sharpe ratio (simplified - assumes daily returns)
    if len(trades_df) > 1:
        returns = trades_df['pnl'] / initial_capital
        sharpe_ratio = returns.mean() / returns.std() * np.sqrt(252) if returns.std() > 0 else 0
    else:
        sharpe_ratio = 0
    
    metrics = {
        'Total Trades': total_trades,
        'Win Rate (%)': win_rate * 100,
        'Winning Trades': winning_trades,
        'Losing Trades': losing_trades,
        'Total P&L ($)': total_pnl,
        'Total Return (%)': total_return * 100,
        'CAGR (%)': cagr * 100,
        'Profit Factor': profit_factor,
        'Avg Win ($)': avg_win,
        'Avg Loss ($)': avg_loss,
        'Max Drawdown (%)': max_drawdown * 100,
        'Sharpe Ratio': sharpe_ratio,
        'Final Capital ($)': final_capital
    }
    
    return metrics

def plot_strategy_results(data, signals, trades, market_name):
    """
    Create comprehensive visualization of strategy results
    """
    fig, axes = plt.subplots(4, 1, figsize=(15, 16))
    
    # 1. Price chart with signals
    axes[0].plot(data['Date'], data['Close'], label='Price', color='black', linewidth=1)
    
    # Mark entry points
    long_signals = signals[signals['Signal'] == 1]
    short_signals = signals[signals['Signal'] == -1]
    
    if len(long_signals) > 0:
        axes[0].scatter(long_signals['Date'], long_signals['Close'], 
                       color='green', marker='^', s=100, label='Long Entry', alpha=0.7)
    
    if len(short_signals) > 0:
        axes[0].scatter(short_signals['Date'], short_signals['Close'], 
                       color='red', marker='v', s=100, label='Short Entry', alpha=0.7)
    
    axes[0].set_title(f'{market_name} - Price and Entry Signals')
    axes[0].set_ylabel('Price')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 2. Commercial Index
    axes[1].plot(data['Date'], data['Commercial_Index'], label='Commercial Index', color='blue')
    axes[1].axhline(y=79, color='green', linestyle='--', alpha=0.7, label='Long Threshold (79)')
    axes[1].axhline(y=21, color='red', linestyle='--', alpha=0.7, label='Short Threshold (21)')
    axes[1].set_title('Commercial Index')
    axes[1].set_ylabel('Index Value')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # 3. RSI
    axes[2].plot(data['Date'], data['RSI'], label='RSI', color='purple')
    axes[2].axhline(y=70, color='red', linestyle='--', alpha=0.7, label='Overbought (70)')
    axes[2].axhline(y=30, color='green', linestyle='--', alpha=0.7, label='Oversold (30)')
    axes[2].set_title('RSI (Relative Strength Index)')
    axes[2].set_ylabel('RSI')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    # 4. Equity Curve
    if trades:
        trades_df = pd.DataFrame(trades)
        equity_curve = [100000]  # Starting capital
        for pnl in trades_df['pnl']:
            equity_curve.append(equity_curve[-1] + pnl)
        
        trade_dates = [trades_df['entry_date'].iloc[0]] + list(trades_df['exit_date'])
        axes[3].plot(pd.to_datetime(trade_dates), equity_curve, label='Equity Curve', color='green', linewidth=2)
        axes[3].axhline(y=100000, color='gray', linestyle='--', alpha=0.5, label='Starting Capital')
        axes[3].set_title('Equity Curve')
        axes[3].set_ylabel('Capital ($)')
        axes[3].legend()
        axes[3].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return fig


In [ ]:
# UPDATED COT + RSI Backtesting Engine with Proper Position Sizing
class COTRSIBacktester:
    """
    Backtesting engine for COT + RSI strategy with 3-day fixed exit
    Updated with proper position sizing and error reporting
    """
    
    def __init__(self, initial_capital=100000, risk_per_trade=0.01, use_stops=True):
        self.initial_capital = initial_capital
        self.risk_per_trade = risk_per_trade  # 1% risk per trade
        self.use_stops = use_stops
        self.trades = []
        self.equity_curve = []
        
    def calculate_position_size(self, entry_price, atr, direction, row_index=None):
        """
        Calculate position size based on 1% risk per trade
        Returns integer units to avoid fractional trades
        
        Formula:
        - Stop price = entry_price - (2 * ATR * direction)
        - Position value = risked_amount / (1 - (stop_price / entry_price))
        - Position units = floor(position_value / entry_price)
        """
        
        # Check for missing ATR data
        if pd.isna(atr) or atr == 0:
            # Allow missing ATR for first 14 days (ATR calculation period)
            if row_index is not None and row_index >= 14:
                raise ValueError(f"❌ MISSING ATR DATA at row {row_index}! "
                               f"Entry price: ${entry_price:.4f}, ATR: {atr}. "
                               f"Check OHLC data quality - High/Low prices may be missing.")
            elif row_index is not None and row_index < 14:
                print(f"⚠ ATR not available for row {row_index} (within first 14 days) - SKIPPING TRADE")
                return None  # Skip this trade
            else:
                raise ValueError(f"❌ MISSING ATR DATA! Entry price: ${entry_price:.4f}, ATR: {atr}. "
                               f"Check if OHLC data (High/Low) is available in your JSON file.")
        
        # Calculate stop price based on direction
        if direction == 1:  # Long position
            stop_price = entry_price - (2 * atr)
        else:  # Short position  
            stop_price = entry_price + (2 * atr)
        
        # Ensure stop price is positive and makes sense
        if stop_price <= 0:
            raise ValueError(f"❌ INVALID STOP PRICE: ${stop_price:.4f} "
                            f"(Entry: ${entry_price:.4f}, ATR: {atr:.4f}, Direction: {direction})")
        
        # Calculate risked amount (1% of capital)
        risked_amount = self.initial_capital * self.risk_per_trade
        
        # Calculate position value using your formula
        risk_ratio = 1 - (stop_price / entry_price)
        
        if risk_ratio <= 0:
            raise ValueError(f"❌ INVALID RISK RATIO: {risk_ratio:.4f} "
                            f"(Entry: ${entry_price:.4f}, Stop: ${stop_price:.4f})")
        
        position_value = risked_amount / risk_ratio
        
        # Calculate integer position units
        position_units = int(position_value / entry_price)
        
        # Ensure minimum 1 unit
        position_units = max(1, position_units)
        
        print(f"✓ Entry: ${entry_price:.4f}, Stop: ${stop_price:.4f}, Risk: ${risked_amount:.2f}, Units: {position_units}")
        
        return position_units
    
    def backtest_strategy(self, data, signals):
        """
        Execute backtest with 3-day fixed exit and optional ATR stops
        """
        self.trades = []
        current_capital = self.initial_capital
        
        i = 0
        while i < len(signals):
            if signals.iloc[i]['Signal'] != 0:
                # Found a signal - enter trade
                entry_signal = signals.iloc[i]
                entry_price = entry_signal['Entry_Price']
                
                if pd.isna(entry_price):
                    i += 1
                    continue
                    
                direction = entry_signal['Signal']  # 1 for long, -1 for short
                atr = entry_signal['ATR']
                
                # Calculate position size with row index for error reporting
                try:
                    position_size = self.calculate_position_size(entry_price, atr, direction, row_index=i)
                    
                    # Skip trade if position_size is None (early days without ATR)
                    if position_size is None:
                        i += 1
                        continue
                        
                except ValueError as e:
                    print(f"❌ Position sizing error at row {i}: {e}")
                    i += 1
                    continue
                
                # Set stop-loss and take-profit levels (optional)
                if self.use_stops and not pd.isna(atr):
                    stop_loss = entry_price - (2 * atr * direction)  # 2 ATR stop
                    take_profit = entry_price + (3 * atr * direction)  # 3 ATR target
                else:
                    stop_loss = None
                    take_profit = None
                
                # Look for exit (3 days later or stop/TP hit)
                exit_idx = min(i + 3, len(data) - 1)  # Exit after 3 days
                exit_price = None
                exit_reason = "3-day limit"
                
                # Check for stop/TP hits during the 3-day period
                if self.use_stops and stop_loss and take_profit:
                    for j in range(i + 1, exit_idx + 1):
                        if j >= len(data):
                            break
                            
                        day_high = data.iloc[j].get('High', data.iloc[j]['Close'])
                        day_low = data.iloc[j].get('Low', data.iloc[j]['Close'])
                        
                        # Check stop-loss hit
                        if direction == 1 and day_low <= stop_loss:  # Long position
                            exit_price = stop_loss
                            exit_reason = "Stop Loss"
                            break
                        elif direction == -1 and day_high >= stop_loss:  # Short position
                            exit_price = stop_loss
                            exit_reason = "Stop Loss"
                            break
                            
                        # Check take-profit hit
                        if direction == 1 and day_high >= take_profit:  # Long position
                            exit_price = take_profit
                            exit_reason = "Take Profit"
                            break
                        elif direction == -1 and day_low <= take_profit:  # Short position
                            exit_price = take_profit
                            exit_reason = "Take Profit"
                            break
                
                # If no stop/TP hit, exit at 3-day limit
                if exit_price is None:
                    if 'Open' in data.columns and exit_idx < len(data):
                        exit_price = data.iloc[exit_idx]['Open']  # Exit at open of 4th day
                    else:
                        exit_price = data.iloc[exit_idx]['Close']
                
                # Calculate trade P&L
                if direction == 1:  # Long trade
                    pnl = (exit_price - entry_price) * position_size
                else:  # Short trade
                    pnl = (entry_price - exit_price) * position_size
                
                # Record trade
                trade = {
                    'entry_date': entry_signal['Date'],
                    'exit_date': data.iloc[exit_idx]['Date'],
                    'direction': 'Long' if direction == 1 else 'Short',
                    'entry_price': entry_price,
                    'exit_price': exit_price,
                    'position_size': position_size,
                    'pnl': pnl,
                    'exit_reason': exit_reason,
                    'commercial_index': entry_signal['Commercial_Index'],
                    'rsi': entry_signal['RSI']
                }
                
                self.trades.append(trade)
                current_capital += pnl
                
                # Record equity point
                self.equity_curve.append({
                    'date': data.iloc[exit_idx]['Date'],
                    'equity': current_capital
                })
                
                # Skip to after exit to avoid overlapping trades
                i = exit_idx + 1
            else:
                i += 1
        
        print(f"Backtest completed: {len(self.trades)} trades executed")
        return self.trades


In [ ]:
# STRATEGY TESTING - Run Complete Backtest
def run_strategy_backtest(market_name, use_stops=True):
    """
    Complete strategy backtest for a single market
    """
    print(f"\\n{'='*60}")
    print(f"BACKTESTING COT + RSI STRATEGY: {market_name}")
    print(f"{'='*60}")
    
    # 1. Prepare data
    print("\\n1. Preparing strategy data...")
    strategy_data = prepare_strategy_data(df, market_name)
    
    if strategy_data.empty:
        print(f"❌ No data available for {market_name}")
        return None
    
    print(f"✓ Data prepared: {len(strategy_data)} daily records")
    
    # 2. Generate signals
    print("\\n2. Generating trading signals...")
    signals = generate_signals(strategy_data)
    
    # 3. Run backtest
    print("\\n3. Running backtest...")
    backtester = COTRSIBacktester(initial_capital=100000, risk_per_trade=0.01, use_stops=use_stops)
    trades = backtester.backtest_strategy(strategy_data, signals)
    
    if not trades:
        print(f"❌ No trades generated for {market_name}")
        return None
    
    # 4. Calculate performance
    print("\\n4. Calculating performance metrics...")
    metrics = calculate_performance_metrics(trades)
    
    # 5. Display results
    print(f"\\n{'='*40}")
    print(f"PERFORMANCE SUMMARY: {market_name}")
    print(f"{'='*40}")
    
    for key, value in metrics.items():
        if isinstance(value, float):
            if '$' in key:
                print(f"{key:<25}: ${value:,.2f}")
            elif '%' in key:
                print(f"{key:<25}: {value:.2f}%")
            else:
                print(f"{key:<25}: {value:.3f}")
        else:
            print(f"{key:<25}: {value}")
    
    # 6. Create visualizations
    print("\\n5. Creating visualizations...")
    plot_strategy_results(strategy_data, signals, trades, market_name)
    
    return {
        'market': market_name,
        'data': strategy_data,
        'signals': signals,
        'trades': trades,
        'metrics': metrics,
        'backtester': backtester
    }


In [ ]:
# EXAMPLE: Test the strategy on a sample market
# Uncomment and run the lines below to test the strategy

# Example 1: Test on Gold
# result = run_strategy_backtest("GOLD - COMMODITY EXCHANGE INC.")

# Example 2: Test on Crude Oil  
# result = run_strategy_backtest("CRUDE OIL, LIGHT SWEET - NEW YORK MERCANTILE EXCHANGE")

# Example 3: Test on Euro
# result = run_strategy_backtest("EURO FX - CHICAGO MERCANTILE EXCHANGE")

print("Strategy implementation complete!")
print("\\nTo test the strategy:")
print("1. First, make sure you've run the updated COT notebook to export OHLC data to JSON")
print("2. Then uncomment one of the example lines above and run it")
print("3. The backtest will show performance metrics and charts")
print("\\nStrategy Rules:")
print("- Long: Commercial Index > 79 AND RSI < 30")
print("- Short: Commercial Index < 21 AND RSI > 70") 
print("- Entry: Next day's open after signal")
print("- Exit: 3 days later (or optional ATR stops)")
print("- Risk: 1% per trade")
